# Chapter 4 — Why Code Execution

Hands-on lab, pushed past a single 8-step-budget snapshot into four real
measurements from [`../code/why_code.py`](../code/why_code.py):

1. The original step-budget benchmark (one budget, one snapshot).
2. Does it generalize? A full SWEEP across budgets 3-24.
3. Tool reuse and dynamic revision (unchanged from the original chapter).
4. A real, measured non-determinism demonstration — not just asserted.

In [1]:
import sys
sys.path.insert(0, "../code")

from why_code import (
    run_benchmark, summarize_benchmark, render_benchmark_table,
    sweep_budgets, render_budget_sweep,
    demo_tool_reuse, HYPOTHETICAL_JSON_TOOL_SCHEMA_FOR_MEAN,
    demo_dynamic_revision, demo_nondeterminism,
)

## 1. The original snapshot: one budget (8), one set of task sizes

In [2]:
ks = [1, 3, 5, 6, 7, 10, 20]
results = run_benchmark(ks, max_steps=8)
print(render_benchmark_table(results))
summary = summarize_benchmark(results)
for approach, stats in summary.items():
    print(f"{approach}: {stats['n_success']}/{stats['n_tasks']} succeeded "
          f"({stats['success_rate']:.0%}), avg steps needed = {stats['avg_steps_needed']:.1f}")

  k | json steps | json ok | code steps | code ok
-------------------------------------------------
  1 |          3 |    True |          2 |    True
  3 |          5 |    True |          2 |    True
  5 |          7 |    True |          2 |    True
  6 |          8 |    True |          2 |    True
  7 |          9 |   False |          2 |    True
 10 |         12 |   False |          2 |    True
 20 |         22 |   False |          2 |    True
json_tool_calls: 4/7 succeeded (57%), avg steps needed = 9.4
code_action: 7/7 succeeded (100%), avg steps needed = 2.0


## 2. Does the 57%-vs-100% gap generalize, or was budget=8 special?

One snapshot (57% vs. 100%) invites an obvious question: is that ratio
specific to `max_steps=8`, or does the pattern hold generally? Sweep the
budget itself, from a tight 3 steps up to a generous 24, holding the same
task sizes fixed.

In [3]:
budget_sweep = sweep_budgets([3, 4, 6, 8, 10, 12, 16, 24], ks)
print(render_budget_sweep(budget_sweep))

budget |  json success rate |  code success rate | max k JSON can fit
---------------------------------------------------------------------
     3 |               14% |              100% |                  1
     4 |               14% |              100% |                  2
     6 |               29% |              100% |                  4
     8 |               57% |              100% |                  6
    10 |               71% |              100% |                  8
    12 |               86% |              100% |                 10
    16 |               86% |              100% |                 14
    24 |              100% |              100% |                 22


Code's success rate is **100% at every single budget tested**, including
the tightest one (3 steps) — because a code action never needs more than 2
steps regardless of task size, so any budget that could run an agent at all
is already enough. JSON tool calling's success rate climbs monotonically
with the budget — 14% -> 29% -> 57% -> 71% -> 86% -> 100% — only reaching
parity with code once the budget (24) is generous enough to fit every task
size in this particular test set. **The "max k JSON can fit" column makes
the mechanism explicit: it's always exactly `budget - 2`**, because JSON
needs `k+2` steps per task; there's nothing probabilistic or emergent about
the curve — it's a direct, exact consequence of the two step-cost formulas
(`k+2` vs. constant `2`). The 8-step snapshot from §1 is one point on this
line, not a cherry-picked special case.

## 3. Tool reuse and dynamic revision (unchanged mechanics)

In [4]:
mean_result = demo_tool_reuse()
print(f"code action result: {mean_result}")
print(f"JSON mode would first need a schema like:\n  {HYPOTHETICAL_JSON_TOOL_SCHEMA_FOR_MEAN}")

traceback_text, fixed_avg = demo_dynamic_revision()
print(f"\nFirst action's real traceback (tail): {traceback_text.strip().splitlines()[-1]}")
print(f"Second action's result after the fix: {fixed_avg}")

code action result: 13.2
JSON mode would first need a schema like:
  {'name': 'compute_mean', 'description': 'Compute the arithmetic mean of a list of numbers.', 'parameters': {'type': 'object', 'properties': {'values': {'type': 'array', 'items': {'type': 'number'}}}, 'required': ['values']}}

First action's real traceback (tail): ZeroDivisionError: division by zero
Second action's result after the fix: 0.0


## 4. Non-determinism — measured, not just claimed

The original chapter asserted "non-determinism" as a cost of code actions
without demonstrating it. Here's a direct demonstration: run the exact same
source text twice and check whether the output differs.

In [5]:
r1, r2 = demo_nondeterminism()
print(f"run 1: {r1}")
print(f"run 2: {r2}")
print(f"identical source code, different output: {r1 != r2}")

run 1: 1784809346.162417
run 2: 1784809346.162428
identical source code, different output: True


Same source text (`import time; result = time.time()`), executed twice,
genuinely different results — not because anything went wrong, but because
`exec()` places no constraint at all on what a code action is allowed to
call. A JSON tool-calling system's non-determinism, by contrast, is bounded
by whatever tools were actually registered — if nobody registered a
`get_current_time` tool, the agent has no way to introduce this specific
non-determinism at all. This is the real, demonstrated shape of the
"wider failure surface" cost side of Chapter 4's argument: not a
hypothetical risk, a one-line reproduction.